# CompBench Metrics (Text Prompts)

Scores saved text-prompt outputs; this notebook does not run the editing agent. It selects the latest CompBench annotation and inference runs by default. Set `MMEDIT_COMPBENCH_ANNOTATION_RUN` or `MMEDIT_COMPBENCH_TEXT_RUN` to select a specific `run_<timestamp>`. Per-sample scores and task/overall means are saved under the selected inference run's `metrics/` directory.

This reproduces the five automated metrics in the [CompBench paper](https://arxiv.org/abs/2505.12200) and [released evaluator](https://github.com/comp-bench/CompBench): LC-T (local CLIP text), LC-I (local CLIP image similarity to the edited ground truth), and background PSNR, SSIM, and LPIPS. Images and masks are resized to 512×512 (bicubic for RGB, nearest-neighbor for masks); LC-T/LC-I use the edit-mask bounding-box crop, while background metrics compare source and generated images with the edited region masked out. Multi-object caption scores are averaged across the pipe-separated captions. The paper also reports separate 0–10 VLM/human judgments for action, location, and viewpoint tasks; those judgments are not generated by these automated metrics.

CompBench annotation files can be sparse: `metadata.row_index` is the original dataset row, not the record position. Saved outputs are matched by `sample_id`.

In [ ]:
import os

from evaluation.notebook import load_benchmark, resolve_run_directory

BENCHMARK = "compbench"
PROMPT_MODE = "text"
context = load_benchmark(BENCHMARK, prompt_mode=PROMPT_MODE)
ANNOTATION_FILE = context.annotation_file
OUTPUT_ROOT = context.output_dir
RUN_NAME = os.environ.get("MMEDIT_COMPBENCH_TEXT_RUN") or None
OUTPUT_DIR = resolve_run_directory(OUTPUT_ROOT, RUN_NAME)
records = context.records
image_loader = context.image_loader
CLIP_MODEL_NAME = os.environ.get(
    "COMPBENCH_CLIP_MODEL", "openai/clip-vit-large-patch14"
)
DEVICE = os.environ.get("COMPBENCH_METRICS_DEVICE") or None
print("Annotations:", ANNOTATION_FILE)
print("Selected run:", OUTPUT_DIR)
print("CLIP model:", CLIP_MODEL_NAME)
print("Metrics device:", DEVICE or "auto")

In [ ]:
from evaluation.compbench_metrics import collect_saved_outputs

saved_outputs = collect_saved_outputs(
    records, OUTPUT_DIR, prompt_mode=PROMPT_MODE
)
print(f"Found {len(saved_outputs)} saved output images:")
for saved in saved_outputs:
    print(f"  [{saved['record_position']}] {saved['sample_id']}")

## Score saved outputs

Install the optional metrics dependencies with `uv sync --extra metrics` in the project environment if needed, then select that environment as the notebook kernel. The first run may download the CLIP model. Scores are cached by output path, size, and modification time; progress is checkpointed every 25 samples.

In [ ]:
from IPython.display import display

from evaluation.compbench_metrics import evaluate_saved_compbench_outputs

metric_rows, score_summary = evaluate_saved_compbench_outputs(
    records,
    output_dir=OUTPUT_DIR,
    image_loader=image_loader,
    prompt_mode=PROMPT_MODE,
    clip_model_name=CLIP_MODEL_NAME,
    device_name=DEVICE,
)
display(
    [
        {"task": task, **metrics}
        for task, metrics in score_summary.items()
    ]
)

In [ ]:
from IPython.display import Markdown, display

CATEGORY_TASKS = {
    "Local": ("add", "remove", "replace"),
    "Multi-object": ("multi_object_add", "multi_object_remove"),
}
METRICS = ("SSIM", "PSNR", "LPIPS", "LC-T", "LC-I")


def weighted_category_summary(tasks):
    task_summaries = [score_summary[task] for task in tasks if task in score_summary]
    if len(task_summaries) != len(tasks):
        missing = sorted(set(tasks) - set(score_summary))
        raise ValueError(f"Missing task summaries: {missing}")
    count = sum(summary["count"] for summary in task_summaries)
    return {
        "count": count,
        **{
            metric: sum(summary[metric] * summary["count"] for summary in task_summaries)
            / count
            for metric in METRICS
        },
    }


local = weighted_category_summary(CATEGORY_TASKS["Local"])
multi_object = weighted_category_summary(CATEGORY_TASKS["Multi-object"])
columns = [
    "Model",
    "Local n",
    "Local SSIM ↑",
    "Local PSNR ↑",
    "Local LPIPS ↓",
    "Local LC-T ↑",
    "Local LC-I ↑",
    "Multi-object n",
    "Multi-object SSIM ↑",
    "Multi-object PSNR ↑",
    "Multi-object LPIPS ↓",
    "Multi-object LC-T ↑",
    "Multi-object LC-I ↑",
]
values = [
    f"MMEdit ({PROMPT_MODE})",
    str(local["count"]),
    *(f"{local[metric]:.4f}" for metric in METRICS),
    str(multi_object["count"]),
    *(f"{multi_object[metric]:.4f}" for metric in METRICS),
]
markdown_table = "| " + " | ".join(columns) + " |\n"
markdown_table += "| " + " | ".join(["---"] * len(columns)) + " |\n"
markdown_table += "| " + " | ".join(values) + " |"
display(Markdown(f"## CompBench table ({PROMPT_MODE})\n\n" + markdown_table))